# Trimmed Kalman Smoother

All functions live in separate modules. Import them here and run the experiments below.

| Module | Contents |
|---|---|
| `projection.py` | `project_capped_simplex` |
| `smoother.py` | `weighted_batch_smoother` |
| `losses.py` | `measurement_losses`, `compute_innovation_losses`, `summarize_innovation_loss_near_jumps` |
| `measurement_trim.py` | `trimmed_kalman_smoother` |
| `innovation_trim.py` | `innovation_trimmed_kalman_smoother`, `warm_start_nu_from_jump_score`, `detection_near_jumps` |
| `simulation.py` | `simulate_constant_velocity_data`, `simulate_general_smooth_signal` |
| `experiments.py` | `run_one_trial` |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from projection       import project_capped_simplex
from smoother         import weighted_batch_smoother
from losses           import (
    measurement_losses,
    compute_innovation_losses,
    summarize_innovation_loss_near_jumps,
)
from measurement_trim import trimmed_kalman_smoother
from innovation_trim  import (
    innovation_trimmed_kalman_smoother,
    warm_start_nu_from_jump_score,
    detection_near_jumps,
)
from simulation       import (
    simulate_constant_velocity_data,
    simulate_general_smooth_signal,
)
from experiments      import run_one_trial

---
## 1. Capped-simplex projection — quick sanity check

In [ ]:
np.random.seed(0)
N, h = 20, 15
y = np.random.randn(N)
w = project_capped_simplex(y, h)

print("Projected weights:", w)
print("sum(w) =", np.sum(w), "  target h =", h)
print("min(w) =", np.min(w), "  max(w) =", np.max(w))

---
## 2. Constant-velocity model — Gaussian smoother baseline

In [ ]:
G, H, Q, R, x_true, z, outlier_idx = simulate_constant_velocity_data()

x_gauss = weighted_batch_smoother(
    G, H, Q, R, z,
    omega=np.ones(len(z)),
    x0_prior=np.array([1, 0]),
    P0=1.0 * np.eye(2),
)

plt.figure(figsize=(12, 5))
plt.plot(x_true[:, 1], label="True position", linewidth=2)
plt.plot(x_gauss[:, 1], label="Gaussian smoother", linewidth=2)
plt.scatter(np.arange(len(z)), z, s=20, alpha=0.6, label="Measurements")
plt.scatter(outlier_idx, z[outlier_idx], s=60, marker="x", label="True outliers")
plt.legend(); plt.title("Gaussian smoother with measurement outliers")
plt.xlabel("Time index"); plt.ylabel("Position"); plt.grid(True)
plt.show()

mse_gauss = np.mean((x_gauss[:, 1] - x_true[:, 1]) ** 2)
print("Gaussian smoother position MSE:", mse_gauss)

---
## 3. Constant-velocity model — measurement-trimmed smoother

In [ ]:
N = len(z)
h = N - len(outlier_idx)   # keep all non-outlier measurements

x_trim, omega_trim, losses_trim, history = trimmed_kalman_smoother(
    G, H, Q, R, z,
    h=h, gamma=0.5, max_iter=50, tol=1e-8,
    x0_prior=np.array([1, 0]),
    P0=1.0 * np.eye(2),
    verbose=True,
)

mse_trim = np.mean((x_trim[:, 1] - x_true[:, 1]) ** 2)
print("\nResults")
print("Gaussian MSE:", mse_gauss)
print("Trimmed MSE:", mse_trim)
print("Improvement factor:", mse_gauss / mse_trim)
print("sum(omega_trim) =", np.sum(omega_trim))

In [ ]:
q_trim = N - h
trimmed_idx = np.argsort(omega_trim)[:q_trim]

plt.figure(figsize=(12, 5))
plt.plot(x_true[:, 1], label="True position", linewidth=2)
plt.plot(x_gauss[:, 1], label="Gaussian smoother", linewidth=2)
plt.plot(x_trim[:, 1], label="Trimmed smoother", linewidth=2)
plt.scatter(np.arange(N), z, s=20, alpha=0.5, label="Measurements")
plt.scatter(outlier_idx, z[outlier_idx], s=70, marker="x", label="True outliers")
plt.scatter(trimmed_idx, z[trimmed_idx], s=120,
            facecolors="none", edgecolors="black", label="Trimmed by algorithm")
plt.legend(); plt.title("Gaussian vs Measurement-Trimmed Kalman Smoother")
plt.xlabel("Time index"); plt.ylabel("Position"); plt.grid(True)
plt.show()

In [ ]:
# Outlier detection metrics
true_outliers    = set(outlier_idx.tolist())
detected_trimmed = set(trimmed_idx.tolist())
correctly_trimmed = true_outliers & detected_trimmed
false_trimmed     = detected_trimmed - true_outliers

print("True outlier indices:    ", sorted(true_outliers))
print("Algorithm-trimmed indices:", sorted(detected_trimmed))
print("Correctly trimmed:        ", sorted(correctly_trimmed))
print("False trimmed:            ", sorted(false_trimmed))
print("Detection rate:", len(correctly_trimmed) / len(true_outliers))

In [ ]:
# Weights and convergence
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

axes[0].stem(np.arange(N), omega_trim)
axes[0].scatter(outlier_idx, omega_trim[outlier_idx], s=80, marker="x", label="True outliers")
axes[0].set_title("Final trimming weights")
axes[0].set_xlabel("Time index"); axes[0].set_ylabel(r"$\omega_k$")
axes[0].set_ylim(-0.05, 1.05); axes[0].grid(True); axes[0].legend()

omega_changes = [item["omega_change"] for item in history]
axes[1].plot(omega_changes, marker="o")
axes[1].set_yscale("log")
axes[1].set_title("Convergence of trimming weights")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel(r"$\|\omega^{\nu+1}-\omega^\nu\|$")
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 4. Monte Carlo experiment (30 trials)

In [ ]:
results = [run_one_trial(seed=s) for s in range(30)]
df = pd.DataFrame(results)
display(df)

cols = ["mse_gaussian", "mse_trimmed", "improvement_factor",
        "detection_rate", "false_trim_rate", "iterations"]
print("\nMeans:")
print(df[cols].mean())
print("\nStd devs:")
print(df[cols].std())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].boxplot([df["mse_gaussian"], df["mse_trimmed"]],
                tick_labels=["Gaussian", "Trimmed"])
axes[0].set_yscale("log")
axes[0].set_title("MSE comparison over 30 trials")
axes[0].set_ylabel("Position MSE"); axes[0].grid(True)

axes[1].hist(df["detection_rate"], bins=10)
axes[1].set_title("Outlier detection rate over 30 trials")
axes[1].set_xlabel("Detection rate"); axes[1].set_ylabel("Count")
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 5. General smooth damped oscillatory signal — setup

True signal: $s(t) = e^{-\beta t} \sin(\alpha t)$

State: $X_k = [\dot{s}_k,\ s_k]^T$

In [ ]:
alpha_general = 4.0
beta_general  = 0.2

(
    t_general, x_true_general, z_general,
    G_general, H_general, Q_general, R_general,
) = simulate_general_smooth_signal(
    N=150, t_max=8.0,
    alpha=alpha_general, beta=beta_general,
    process_scale=2.0, meas_std=0.15, seed=1,
)

N_general = len(z_general)
x0_prior_general = np.array([alpha_general, 0.0])

plt.figure(figsize=(12, 5))
plt.plot(t_general, x_true_general[:, 1], linewidth=2, label="True signal")
plt.scatter(t_general, z_general, s=20, alpha=0.55, label="Noisy measurements")
plt.xlabel("Time"); plt.ylabel("Signal value")
plt.title("General smooth damped oscillatory signal")
plt.legend(); plt.grid(alpha=0.25); plt.show()

In [ ]:
# Gaussian smoother on clean signal
x_gaussian_general = weighted_batch_smoother(
    G=G_general, H=H_general, Q=Q_general, R=R_general, z=z_general,
    omega=np.ones(N_general),
    x0_prior=x0_prior_general, P0=np.eye(2),
)

print("Derivative MSE:", np.mean((x_gaussian_general[:, 0] - x_true_general[:, 0])**2))
print("Signal MSE:    ", np.mean((x_gaussian_general[:, 1] - x_true_general[:, 1])**2))

---
## 6. General signal — measurement outlier experiment

In [ ]:
z_general_outlier  = z_general.copy()
outlier_idx_general = np.array([18, 42, 67, 91, 116, 138])
outlier_values_general = np.array([2.8, -3.0, 2.5, -2.7, 3.2, -2.9])
z_general_outlier[outlier_idx_general] += outlier_values_general

# Gaussian smoother with outliers
x_gaussian_general_outlier = weighted_batch_smoother(
    G=G_general, H=H_general, Q=Q_general, R=R_general, z=z_general_outlier,
    omega=np.ones(N_general),
    x0_prior=x0_prior_general, P0=np.eye(2),
)

mse_gaussian_general_outlier_signal = np.mean(
    (x_gaussian_general_outlier[:, 1] - x_true_general[:, 1])**2
)
print("Gaussian signal MSE with outliers:", mse_gaussian_general_outlier_signal)

In [ ]:
# Measurement-trimmed smoother on general signal with outliers
h_general = N_general - len(outlier_idx_general)

(
    x_trimmed_general_outlier,
    omega_trimmed_general_outlier,
    losses_trimmed_general_outlier,
    history_general_outlier,
) = trimmed_kalman_smoother(
    G=G_general, H=H_general, Q=Q_general, R=R_general,
    z=z_general_outlier,
    h=h_general, gamma=0.5, max_iter=100, tol=1e-8,
    x0_prior=x0_prior_general, P0=np.eye(2),
    verbose=True,
)

num_trimmed_general = N_general - h_general
detected_trimmed_idx_general = np.argsort(omega_trimmed_general_outlier)[:num_trimmed_general]

mse_trimmed_general_outlier_signal = np.mean(
    (x_trimmed_general_outlier[:, 1] - x_true_general[:, 1])**2
)

print("\nTrue outlier indices:   ", np.sort(outlier_idx_general))
print("Detected trimmed indices:", np.sort(detected_trimmed_idx_general))
print("Gaussian signal MSE:    ", mse_gaussian_general_outlier_signal)
print("Trimmed signal MSE:     ", mse_trimmed_general_outlier_signal)
print("Improvement factor:     ",
      mse_gaussian_general_outlier_signal / mse_trimmed_general_outlier_signal)

In [ ]:
plt.figure(figsize=(13, 6))
plt.plot(t_general, x_true_general[:, 1], linewidth=2, label="True signal")
plt.scatter(t_general, z_general_outlier, s=18, alpha=0.4, label="Measurements")
plt.scatter(t_general[outlier_idx_general], z_general_outlier[outlier_idx_general],
            marker="x", s=100, linewidths=2, label="True measurement outliers")
plt.plot(t_general, x_gaussian_general_outlier[:, 1],
         linestyle="--", linewidth=2, label="Gaussian smoother")
plt.plot(t_general, x_trimmed_general_outlier[:, 1],
         linestyle="-.", linewidth=2, label="Trimmed smoother")
plt.scatter(t_general[detected_trimmed_idx_general],
            z_general_outlier[detected_trimmed_idx_general],
            facecolors="none", edgecolors="black", s=130, linewidths=1.5,
            label="Measurements selected for trimming")
plt.xlabel("Time"); plt.ylabel("Signal value")
plt.title("General smooth signal: Gaussian versus trimmed smoother")
plt.legend(); plt.grid(alpha=0.25); plt.show()

---
## 7. General signal — genuine state-jump experiment

In [ ]:
# Inject permanent signal-level jumps into the true state
jump_idx_general    = np.array([30, 80, 125])
jump_values_general = np.array([0.9, -1.2, 0.8])

x_true_general_jump = x_true_general.copy()
for jump_idx, jump_value in zip(jump_idx_general, jump_values_general):
    x_true_general_jump[jump_idx:, 1] += jump_value

# Reuse the same measurement-noise realization
measurement_noise_general = z_general - x_true_general[:, 1]
z_general_jump = x_true_general_jump[:, 1] + measurement_noise_general

print("x_true_general_jump shape:", x_true_general_jump.shape)
print("z_general_jump shape:     ", z_general_jump.shape)
print("Jump indices:", jump_idx_general)
print("Jump values: ", jump_values_general)

In [ ]:
# Gaussian smoother on jumped signal
x_gaussian_general_jump = weighted_batch_smoother(
    G=G_general, H=H_general, Q=Q_general, R=R_general, z=z_general_jump,
    omega=np.ones(N_general),
    x0_prior=x0_prior_general, P0=np.eye(2),
)

mse_gaussian_general_jump_signal = np.mean(
    (x_gaussian_general_jump[:, 1] - x_true_general_jump[:, 1])**2
)
print("Gaussian signal MSE with state jumps:", mse_gaussian_general_jump_signal)

In [ ]:
# Innovation losses: compare Gaussian vs measurement-trimmed
innovation_residuals_gaussian_jump, innovation_losses_gaussian_jump = \
    compute_innovation_losses(x_gaussian_general_jump, G_general, Q_general)

# Measurement-trimmed on jumped signal (no true measurement outliers)
assumed_num_outliers_general_jump = len(outlier_idx_general)
h_general_jump = N_general - assumed_num_outliers_general_jump

(
    x_trimmed_general_jump,
    omega_trimmed_general_jump,
    losses_trimmed_general_jump,
    history_general_jump,
) = trimmed_kalman_smoother(
    G=G_general, H=H_general, Q=Q_general, R=R_general,
    z=z_general_jump,
    h=h_general_jump, gamma=0.5, max_iter=100, tol=1e-8,
    x0_prior=x0_prior_general, P0=np.eye(2),
    verbose=True,
)

innovation_residuals_trimmed_jump, innovation_losses_trimmed_jump = \
    compute_innovation_losses(x_trimmed_general_jump, G_general, Q_general)

summary_gaussian = summarize_innovation_loss_near_jumps(
    innovation_losses_gaussian_jump, jump_idx_general, radius=3)
summary_trimmed  = summarize_innovation_loss_near_jumps(
    innovation_losses_trimmed_jump, jump_idx_general, radius=3)

print("Gaussian innovation-loss summary near jumps")
display(summary_gaussian)
print("Trimmed innovation-loss summary near jumps")
display(summary_trimmed)

---
## 8. Innovation-trimmed smoother — jump detection

In [ ]:
q_innovation              = len(jump_idx_general)
h_innovation_general_jump = (N_general - 1) - q_innovation

(
    x_innovation_trimmed_general_jump,
    nu_innovation_trimmed_general_jump,
    innovation_losses_innovation_trimmed_jump,
    innovation_residuals_innovation_trimmed_jump,
    history_innovation_trimmed_jump,
) = innovation_trimmed_kalman_smoother(
    G=G_general, H=H_general, Q=Q_general, R=R_general,
    z=z_general_jump,
    h_innovation=h_innovation_general_jump,
    gamma=None, gamma_factor=1.0,
    max_iter=100, tol=1e-8,
    x0_prior=x0_prior_general, P0=np.eye(2),
    verbose=True,
)

In [ ]:
true_jump_transition_idx_general = jump_idx_general - 1

detected_innovation_trimmed_idx = np.argsort(
    nu_innovation_trimmed_general_jump
)[:q_innovation]

print("True jump transition indices:", true_jump_transition_idx_general)
print("Innovation transitions selected for trimming:",
      np.sort(detected_innovation_trimmed_idx))

innovation_detection_rate, innovation_matches = detection_near_jumps(
    detected_innovation_trimmed_idx,
    true_jump_transition_idx_general,
    radius=2,
)
print("\nInnovation-jump detection rate:", innovation_detection_rate)
print("Matched each jump?", innovation_matches)

In [ ]:
# Plot innovation weights along transition timeline
transition_times_general = 0.5 * (t_general[:-1] + t_general[1:])

plt.figure(figsize=(13, 4))
plt.plot(transition_times_general, nu_innovation_trimmed_general_jump,
         marker="o", markersize=3, linewidth=1.5, label="Innovation weight nu")
for i, transition_idx in enumerate(true_jump_transition_idx_general):
    plt.axvline(transition_times_general[transition_idx], linestyle=":", linewidth=2,
                label="True jump transition" if i == 0 else None)
plt.axhline(0.5, linestyle="--", linewidth=1, label="Weight = 0.5")
plt.xlabel("Transition time"); plt.ylabel("Innovation weight")
plt.title("Innovation-trimming weights near genuine state jumps")
plt.ylim(-0.05, 1.05); plt.legend(); plt.grid(alpha=0.25); plt.show()

---
## 9. Innovation trimming with warm start

In [ ]:
L = len(z_general_jump) - 1
h_innovation = L - q_innovation

nu_init, warm_idx, jump_score = warm_start_nu_from_jump_score(
    z_general_jump, h_innovation=h_innovation, small_value=0.1
)

print("Warm-start suspected jump transitions:", warm_idx)
print("True jump transitions:                ", jump_idx_general - 1)

In [ ]:
(
    x_innov_warm, nu_innov_warm,
    innov_losses_warm, innov_residuals_warm, history_warm,
) = innovation_trimmed_kalman_smoother(
    G=G_general, H=H_general, Q=Q_general, R=R_general,
    z=z_general_jump,
    h_innovation=h_innovation,
    gamma=None, gamma_factor=0.25,
    max_iter=100, tol=1e-8,
    x0_prior=x0_prior_general, P0=np.eye(2),
    nu_init=nu_init, verbose=True,
)